# Some notes

## What to know (concepts)
#### https://www.stephendiehl.com/posts/mlir_introduction/

A modern compiler weaves these components together to make up a pipeline:

-Source code is parsed into a surface language.

-Surface language is translated to a core language.

-Core langauge can be optimized by core-to-core transformations (optionally using tools like e-graphs).

-Core language is lowered to MLIR.

-MLIR performs high-level optimizations

-MLIR is lowered to LLVM IR

-LLVM performs low-level optimizations and code generation


## IR anatomy

#### Modules: The top-level container for MLIR operations.

module {
  // Operations
}

#### Functions: A function is a collection of operations that are executed in a specific order.

#### Operation: The basic unit of work in MLIR, similar to LLVM instructions but with an additional dialect namespace and possible type annotations. It has a name (dialect prefix + op), operands, results, attributes, types, and may contain regions → blocks → operations again.
// Example of an MLIR operation
%0 = "my_dialect.my_operation"(%arg0, %arg1) : (i32, i32) -> i32
In this example, my_dialect.my_operation is an operation defined in the my_dialect dialect, and %arg0 and %arg1 are arguments of type i32. It returns a result of type i32 bound to the variable %0.

A concrete example would be the "arith.addf" operation, which adds two floating point numbers.

%0 = arith.addf %arg0, %arg1 : f32

Each op prints/parses in MLIR’s textual form; you’ll be transforming this textually visible IR.

#### Basic Blocks: Basic blocks in MLIR are sequences of operations that execute in a linear fashion without any branching. 
Each basic block has a single entry point and can have multiple exit points, typically through control flow operations like branches or returns. Basic blocks are essential for structuring the flow of control in a program, allowing for clear and maintainable code organization. They enable the compiler to optimize the execution path and facilitate transformations such as inlining and loop unrolling. In MLIR, basic blocks are defined within functions and can be manipulated through various dialects to represent complex control flow constructs.

^bb1: // Label for the then block
%then_result = arith.muli %result, 2 : i32
return %then_result : i32

Unike in LLVM, basic blocks can now also take arguments, which are passed in through the ^bb1(%result : i32) syntax.

#### Regions: Regions are a way to group operations together. 
They are used to represent control flow constructs like loops and conditionals. Regions are grouped together in {} blocks and can take arguments.

{
  ^bb1(%result : i32):
  %then_result = arith.muli %result, 2 : i32
  return %then_result : i32
}

#### Types: A type is a classification that specifies which kind of value a variable can hold and what operations can be performed on it. 
Types help to enforce constraints on data, ensuring that operations are performed on compatible values.

%result = arith.constant 1 : i32
We can also defined type synonyms for convenience.

!avx_m128 = vector<4 x f32>

## Identifiers
First some identifier conventions:

% prefix: SSA values (e.g. %result)

@ prefix: Functions (e.g. @fun)

^ prefix: Basic blocks (e.g. ^bb0)

\#/ prefix: Attribute aliases (e.g. #map_1d_identity)

x delimiter: Used in shapes and between shape and type (e.g. 10xf32)

: and -> are used to indicate the type of an operation or value (e.g. %result: i32)

! prefix: Type aliases (e.g. !avx_m128 = vector<4 x f32>)

() are used to represent arguments (e.g. (%arg0, %arg1))

{} are used to represent regions

// is used for comments

<> are used to indicate type parameters (e.g. tensor<10xf32>)


## Dialects : Domain-specific sets of operations and types in MLIR.

 
The dialect system in MLIR can represent and optimize operations at various levels, from high-level ML tasks down to hardware-specific instructions. This means we can implement optimizations tailored to specific domains, making AI workloads run more efficiently.Instead of needing separate compilers for each type of accelerator or framework, MLIR offers a unified infrastructure that can be extended for different use cases. Traditional compilers weren’t built with AI in mind, but MLIR has first-class support for tensor operations, neural networks, transformers, and other ML constructs.MLIR has become the go-to tech for specialized machine learning accelerators, finding applications in signal processing, quantum computing, homomorphic encryption, FPGAs, and custom silicon. Its ability to create domain-specific compilers is a game-changer for those "weird domains" that don’t fit the traditional CPU and GPU mold. Plus, it can be easily embedded in other languages, allowing us to build domain-specific languages tailored for specific tasks

### The high-level ones well use are:

tensor: This dialect provides operations for creating and manipulating multi-dimensional arrays (tensors), enabling high-level mathematical expressions and transformations in a side-effect-free manner.

linalg: The linear algebra (linalg) dialect offers a set of operations specifically designed for linear algebra computations, facilitating efficient implementations of matrix and vector operations.

omp: The OpenMP (omp) dialect includes operations that support parallel programming models, allowing developers to express parallelism in a way that is compatible with OpenMP standards.

affine: The affine dialect provides a framework for expressing affine operations and analyses, enabling optimizations that leverage the predictable behavior of affine expressions.

gpu: The GPU dialect is specialized for GPU programming, providing operations for parallel execution on heterogeneous architectures.


### The lower-level ones are:

scf: The structured control flow (scf) dialect provides operations that represent structured control flow constructs, such as loops and conditionals, ensuring a clear and maintainable flow of execution.

func: This dialect encompasses operations related to function definitions and calls, facilitating the organization and modularization of code through high-level function abstractions.

memref: The memref dialect is dedicated to memory reference operations, allowing for efficient management and manipulation of memory buffers, essential for performance-critical applications.

index: The index dialect specializes in operations for handling index computations, which are crucial for addressing elements in arrays and tensors, particularly in loop iterations and data access patterns.

arith: The arithmetic (arith) dialect contains fundamental mathematical operations for both integer and floating-point types, including basic arithmetic, bitwise operations, and comparisons, applicable to scalars, vectors, and tensors. (e.g., arith.addi/addf, arith.muli/mulf, arith.constant)

--->> Know how to add a dialect to a registry if you run locally.

## Passes vs. Patterns

#### Passes: Transformations that operate on the MLIR dialects, optimizing and lowering them into simpler constructs.
A Pass chooses the scope (Module/Func/Op) and orchestrates rewrites.

These are arguments that can be passed to mlir-opt to transform the MLIR, the most common ones are:

convert-func-to-llvm: Convert function-like operations to LLVM dialect

convert-math-to-llvm: Convert math operations to LLVM dialect

convert-index-to-llvm: Convert index operations to LLVM dialect

convert-scf-to-cf: Convert structured control flow to CF dialect

convert-cf-to-llvm: Convert control flow to LLVM dialect

convert-arith-to-llvm: Convert arithmetic operations to LLVM dialect

reconcile-unrealized-casts: Reconcile unrealized casts

convert-memref-to-llvm: Convert memref operations to LLVM dialect

convert-tensor-to-llvm: Convert tensor operations to LLVM dialect

convert-linalg-to-scf: Convert linalg operations to scf.for loops

convert-linalg-to-affine-loops: Convert linalg operations to affine.for loops

convert-omp-to-llvm: Convert OpenMP operations to LLVM dialect

convert-vector-to-llvm: Convert vector operations to LLVM dialect

There is also a generic -convert-to-llvm pass that will convert anything that can be converted from MLRI to LLVM IR. In practice we'll use more granular passes to convert specific dialects to LLVM.

If you need more granular control over the passes you can also specify the pass names in a comma-separated list in a --pass-pipeline string, e.g., --pass-pipeline="builtin.module(pass1,pass2)". The passes will be run sequentially in one group.

Additionally there are several use flags for debugging pass transformation. The --mlir-print-ir-after-all flag prints the IR after each pass, while --mlir-print-ir-after-change and --mlir-print-ir-after-failure provide more specific output. When using any of these print-ir flags, including --mlir-print-ir-tree-dir, the IRs are written to files in a directory tree if you don't want to parse through the terminal stdout.

Note: The order of the passes can be important, for example convert-scf-to-cf must come before convert-cf-to-llvm.
Rewrite Patterns (classes that inherit OpRewritePattern<...>) do the actual match + rewrite.

Use applyPatternsAndFoldGreedily (or the “GreedyPatternRewriteDriver”) to repeatedly apply your patterns + canonical folds until fixpoint.

### Matching & rewriting

matchAndRewrite(op, rewriter) is the workhorse.

Use op.getOperand(i), op.getResult(i), getDefiningOp<arith::ConstantOp>(), etc.

Replace with rewriter.replaceOp(op, newValue) or build new ops with rewriter.create<...>(loc, …).

Delete dead ops with rewriter.eraseOp(op) (only when safe).

Constant folding often falls out of applyPatternsAndFoldGreedily if you create constant ops or call canonical ops.

### Safety / correctness details

Float rules: x + 0.0 is usually safe, but -0.0 and NaN edge cases can break semantics. If they gave you float examples, either:

restrict to integer ops (addi/muli) or

document assumptions (e.g., “assume no NaNs / -0.0 in inputs”) if that’s stated in problem text.

Types must match when creating constants or replacing values.

Prefer using the canonicalization that already exists (e.g., many arith ops have built-in folds); your custom rules should handle the specific patterns they asked for.

### Typical simplifications

Algebraic identities: x + 0 → x, x * 1 → x, x * 0 → 0, x - 0 → x, x / 1 → x

Constant folding: (const a) + (const b) → const (a+b)

Strength reduction: x * 2 → x + x (only if they ask)

Dead code elimination: remove ops with no uses (often handled by -eliminate-dead-code or greedy folding if you erase after replacement)

Common subexpression elimination: usually a separate pass (-cse), don’t re-implement unless requested.

### Pass registration and running

Register your pass with a factory (e.g., std::unique_ptr<Pass> createSimplifyPass();) and PassPipelineRegistration if you want a pipeline flag.



## Minimal mental model (how your code fits together)

### Collect patterns

RewritePatternSet patterns(&getContext());
patterns.add<MyRule1, MyRule2>(&getContext());
if (failed(applyPatternsAndFoldGreedily(getOperation(), std::move(patterns))))
  return signalPassFailure();


### A typical pattern

struct AddZeroSimplify : OpRewritePattern<arith::AddIOp> {
  using OpRewritePattern::OpRewritePattern;
  LogicalResult matchAndRewrite(arith::AddIOp op,
                                PatternRewriter &rewriter) const override {
    Value lhs = op.getLhs(), rhs = op.getRhs();

    // Match constant 0 on either side
    auto isZero = [](Value v) {
      if (auto c = v.getDefiningOp<arith::ConstantOp>())
        if (auto i = dyn_cast<IntegerAttr>(c.getValue()))
          return i.getValue().isZero();
      return false;
    };

    if (isZero(rhs)) { rewriter.replaceOp(op, lhs); return success(); }
    if (isZero(lhs)) { rewriter.replaceOp(op, rhs); return success(); }
    return failure();
  }
};


Adapt similarly for muli, addi with constants, etc. For floats, be careful (see “Safety”).

## How to handle transforms efficiently (step-by-step)

1) Skim the provided input → output examples first.
Underline exact transforms they expect. That becomes your checklist of patterns.

2) Write a pattern list.
For each expected transform, write the op + the condition. Example:

arith.addi %x, (arith.constant 0) → %x

arith.muli %x, (arith.constant 1) → %x

(arith.constant a) + (arith.constant b) → (arith.constant a+b) (same for mul)

3) Implement patterns one by one.

Start with the simplest identity (+0, *1, *0).

Then constant folding (two constants).

Run after each pattern to confirm the output matches the expected IR.

4) Use greedy driver + built-in folding.

Always wrap your patterns with applyPatternsAndFoldGreedily.
This often removes now-unused constants and simplifies nested expressions automatically.

5) Add debug aids (as needed).

Use rewriter.notifyMatchFailure(op, "reason"); while developing.

If you get crashes, check types and locations (op.getLoc() when creating new constants).

6) Final pass polish.

Ensure you added patterns for both operand orders if commutative (e.g., 0 + x and x + 0).

Keep patterns small & deterministic. Avoid creating cycles (e.g., don’t rewrite in ways that cause infinite toggling).

### Pitfalls & quick fixes

Wrong dialect includes or missing registration → add the dialects you use to the registry if you build locally.

Type mismatch for constants → when creating arith::ConstantOp, use the original operand’s element/type.

Float edge cases → if the spec doesn’t require float correctness with NaNs/−0.0, either skip float rules or guard them with comments/assumptions.

Pattern not firing → verify the exact op class (arith::AddIOp vs arith::AddFOp), operand order, and that your isZero() or isOne() matches the same integer width/type.

Testing workflow (even in their browser runner)

Create tiny IR snippets that isolate each rule:

%c0 = arith.constant 0 : i32
%x1 = arith.addi %arg0, %c0 : i32
%x2 = arith.addi %c0, %arg0 : i32
%c2 = arith.constant 2 : i32
%c3 = arith.constant 3 : i32
%x3 = arith.addi %c2, %c3 : i32


After your pass, verify they become %arg0, %arg0, and arith.constant 5 : i32, etc.

If provided expected output, compare line-by-line. Your pass doesn’t need to be “smart,” just exactly produce the expected form.

If you also want to be ready to run locally (optional)

Link against: MLIRIR, MLIRPass, MLIRTransforms, MLIRRewrite, MLIRArithDialect, MLIRFuncDialect, MLIRParser, MLIRSupport, (and MLIROptLib if you build a standalone mlir-opt-like tool).

In your tool’s main, create a DialectRegistry, insert the dialects you need, then call MlirOptMain with your registered pass pipeline.

## Writing MLIR For example writing loops in LLVM is actually a bit annoying as you have to manually handle blocks, phi nodes, all for a basic loop. 
## MLIR allows us to write high-level constructs directly and have them lowered down into LLVM in a single pass.

First, create a file named example.mlir with this MLIR:

func.func @loop_add() -> (index) {
    %init = index.constant 0
    %lb = index.constant 0
    %ub = index.constant 10
    %step = index.constant 1

    %sum = scf.for %iv = %lb to %ub step %step iter_args(%acc = %init) -> (index) {
        %sum_next = arith.addi %acc, %iv : index
        scf.yield %sum_next : index
    }

    return %sum : index
}

func.func @main() -> i32 {
    %out = call @loop_add() : () -> index
    %out_i32 = arith.index_cast %out : index to i32
    func.return %out_i32 : i32
}

### In C this is equivalent to:

int loop_add(int lb, int ub, int step) {
  int sum_0 = 0;
  int sum = sum_0;
  for (int iv = lb; iv < ub; iv += step) {
    sum_next = sum + iv;
    sum = sum_next;
  }
  return sum;
}

int main() {
  int out = loop_add(0, 10, 1);
  return out;
}

## Lower the high-level MLIR to LLVM dialect using mlir-opt.

mlir-opt example.mlir \
  --convert-func-to-llvm \
  --convert-math-to-llvm \
  --convert-index-to-llvm \
  --convert-scf-to-cf \
  --convert-cf-to-llvm \
  --convert-arith-to-llvm \
  --reconcile-unrealized-casts \
  -o example_opt.mlir

## This will produce the following MLIR:

module {
  llvm.func @loop_add() -> i64 {
    %0 = llvm.mlir.constant(0 : i64) : i64
    %1 = llvm.mlir.constant(0 : i64) : i64
    %2 = llvm.mlir.constant(10 : i64) : i64
    %3 = llvm.mlir.constant(1 : i64) : i64
    llvm.br ^bb1(%1, %0 : i64, i64)
  ^bb1(%4: i64, %5: i64):  // 2 preds: ^bb0, ^bb2
    %6 = llvm.icmp "slt" %4, %2 : i64
    llvm.cond_br %6, ^bb2, ^bb3
  ^bb2:  // pred: ^bb1
    %7 = llvm.add %5, %4 : i64
    %8 = llvm.add %4, %3 : i64
    llvm.br ^bb1(%8, %7 : i64, i64)
  ^bb3:  // pred: ^bb1
    llvm.return %5 : i64
  }
  llvm.func @main() -> i32 {
    %0 = llvm.call @loop_add() : () -> i64
    %1 = llvm.trunc %0 : i64 to i32
    llvm.return %1 : i32
  }
}

## We can use the mlir-cpu-runner to run the main function directly from MLIR code.

mlir-cpu-runner -e main -entry-point-result=i32 simple_opt.mlir

## To use the runner utils (e.g. for debug printing)
## mlir-cpu-runner -e main -entry-point-result=i32 -shared-libs=/opt/homebrew/opt/llvm/lib/libmlir_runner_utils.dylib simple_opt.mlir

## Compiling to a Shared Object
## From there is a direct translation of the MLIR to LLVM IR using mlir-translate. We can compile this MLIR to a shared object using mlir-translate:

mlir-translate simple_opt.mlir -mlir-to-llvmir -o simple.ll

llc -filetype=obj --relocation-model=pic simple.ll -o simple.o

clang -shared -fPIC simple.o -o libsimple.so

The LLVM them compiles down into assembly (in this case ARM64 assembly).

_loop_add:
sub     sp, sp, #0x30
mov     x8, #0x0
mov     x9, x8
str     x9, [sp, #0x20]
str     x8, [sp, #0x28]
b       0x100003f10
ldr     x9, [sp, #0x20]
ldr     x8, [sp, #0x28]
str     x8, [sp, #0x8]
str     x9, [sp, #0x10]
subs    x9, x9, #0xa
str     x8, [sp, #0x18]
b.ge    0x100003f4c
b       0x100003f30
ldr     x9, [sp, #0x10]
ldr     x8, [sp, #0x8]
add     x8, x8, x9
add     x9, x9, #0x1
str     x9, [sp, #0x20]
str     x8, [sp, #0x28]
b       0x100003f10
ldr     x0, [sp, #0x18]
add     sp, sp, #0x30
ret


  
  

# Expressing operations in MLIR
## https://www.stephendiehl.com/posts/mlir_affine/
## https://www.stephendiehl.com/posts/mlir_linear_algebra/
Matrix Multiplication
The classic matrix product of two matrices:
Cik= summed j=1 to N(Aij * Bjk)

Expressing this in MLIR as an explicit loop nest would look like this:

func.func @matmul(%A: memref<?x?xf32>, %B: memref<?x?xf32>, %C: memref<?x?xf32>) {
  %c0 = arith.constant 0 : index
  %c1 = arith.constant 1 : index

  %M = memref.dim %A, %c0 : memref<?x?xf32>
  %N = memref.dim %B, %c1 : memref<?x?xf32>
  %K = memref.dim %A, %c1 : memref<?x?xf32>

  // Sequential implementation
  affine.for %i = 0 to %M {
    affine.for %j = 0 to %N {
      affine.for %k = 0 to %K {
        %a = affine.load %A[%i, %k] : memref<?x?xf32>
        %b = affine.load %B[%k, %j] : memref<?x?xf32>
        %c = affine.load %C[%i, %j] : memref<?x?xf32>
        %prod = arith.mulf %a, %b : f32
        %sum = arith.addf %c, %prod : f32
        affine.store %sum, %C[%i, %j] : memref<?x?xf32>
      }
    }
  }
  return
}

Notice that the two loops over %i and %j have no dependencies between different iterations of these loops. Using the -affine-parallelize pass, this can be transformed into:

func.func @matmul_parallel(%A: memref<?x?xf32>, %B: memref<?x?xf32>, %C: memref<?x?xf32>) {
  %c0 = arith.constant 0 : index
  %c1 = arith.constant 1 : index

  %M = memref.dim %A, %c0 : memref<?x?xf32>
  %N = memref.dim %B, %c1 : memref<?x?xf32>
  %K = memref.dim %A, %c1 : memref<?x?xf32>

  // Parallel implementation
  affine.parallel (%i, %j) = (0, 0) to (%M, %N) {
    affine.for %k = 0 to %K {
      %a = affine.load %A[%i, %k] : memref<?x?xf32>
      %b = affine.load %B[%k, %j] : memref<?x?xf32>
      %c = affine.load %C[%i, %j] : memref<?x?xf32>
      %prod = arith.mulf %a, %b : f32
      %sum = arith.addf %c, %prod : f32
      affine.store %sum, %C[%i, %j] : memref<?x?xf32>
    }
  }
  return
}

# Excellent NN: https://www.stephendiehl.com/posts/mlir_neural_networks/
# Excellent transformer: https://www.stephendiehl.com/posts/mlir_transformers/